In [1]:
import imblearn
print(f"imbalanced-learn version: {imblearn.__version__}")

imbalanced-learn version: 0.14.2


In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, 
    fbeta_score, confusion_matrix, precision_recall_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

# Reload final 30-feature state
X_train_final = np.load('../data/X_train_final_30.npy')
X_test_final = np.load('../data/X_test_final_30.npy')
final_feature_names = np.load('../data/final_feature_names_30.npy', allow_pickle=True)

df_train = pd.read_csv('../data/df_train_v2.csv')
df_test = pd.read_csv('../data/df_test_v2.csv')
y_train = df_train['target']
y_test = df_test['target']

print(f"X_train: {X_train_final.shape}")
print(f"X_test:  {X_test_final.shape}")
print(f"y_train: {y_train.shape}, positive rate: {y_train.mean()*100:.2f}%")
print(f"y_test:  {y_test.shape}, positive rate: {y_test.mean()*100:.2f}%")

X_train: (78283, 30)
X_test:  (19539, 30)
y_train: (78283,), positive rate: 11.33%
y_test:  (19539,), positive rate: 11.96%


In [2]:
def evaluate_strategy(name, X_train, y_train, X_test, y_test, resampler=None, model_kwargs=None):
    """
    Evaluate an imbalance-handling strategy on the 30-feature model.
    
    Args:
        name: human-readable label
        resampler: an imblearn object (e.g., SMOTE) or None for no resampling
        model_kwargs: dict of LogisticRegression parameters
    """
    if model_kwargs is None:
        model_kwargs = {}
    
    # Apply resampling to TRAINING data only — test stays original
    if resampler is not None:
        X_train_resampled, y_train_resampled = resampler.fit_resample(X_train, y_train)
    else:
        X_train_resampled, y_train_resampled = X_train, y_train
    
    # Train the model
    model = LogisticRegression(
        penalty='l2',
        solver='lbfgs',
        max_iter=1000,
        random_state=42,
        **model_kwargs
    )
    model.fit(X_train_resampled, y_train_resampled)
    
    # Evaluate on the ORIGINAL test set
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    # Compute metrics
    auc = roc_auc_score(y_test, y_proba)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f2 = fbeta_score(y_test, y_pred, beta=2)
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    return {
        'method': name,
        'train_size': len(y_train_resampled),
        'train_pos_rate': y_train_resampled.mean(),
        'auc': auc,
        'precision': precision,
        'recall': recall,
        'f2': f2,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
        'y_proba': y_proba  # saved for PR curve plotting
    }

print("Evaluation function ready.")

Evaluation function ready.


In [3]:
# Method 1: Current baseline — class_weight='balanced', no resampling
result_baseline = evaluate_strategy(
    name="class_weight=balanced (current)",
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    resampler=None,
    model_kwargs={'class_weight': 'balanced'}
)

# Method 2: SMOTE — synthetic oversampling
result_smote = evaluate_strategy(
    name="SMOTE",
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    resampler=SMOTE(random_state=42),
    model_kwargs={}  # no class_weight needed; resampling handles imbalance
)

# Method 3: Random undersampling
result_undersample = evaluate_strategy(
    name="RandomUnderSampler",
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    resampler=RandomUnderSampler(random_state=42),
    model_kwargs={}
)

# Method 4: SMOTE + Tomek cleaning
result_smote_tomek = evaluate_strategy(
    name="SMOTE + TomekLinks",
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    resampler=SMOTETomek(random_state=42),
    model_kwargs={}
)

# Method 5: No imbalance handling (for reference — predict-everything-negative case)
result_naive = evaluate_strategy(
    name="No imbalance handling (naive)",
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    resampler=None,
    model_kwargs={}  # no class_weight, no resampling
)

# Collect into a comparison table
results = [result_baseline, result_smote, result_undersample, result_smote_tomek, result_naive]
comparison_df = pd.DataFrame([
    {
        'Method': r['method'],
        'Train size': r['train_size'],
        'Train pos%': f"{r['train_pos_rate']*100:.1f}%",
        'AUC': round(r['auc'], 4),
        'Precision': round(r['precision'], 4),
        'Recall': round(r['recall'], 4),
        'F2': round(r['f2'], 4),
        'TP': r['tp'],
        'FN': r['fn']
    } for r in results
])

print(comparison_df.to_string(index=False))

/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in versi

                         Method  Train size Train pos%    AUC  Precision  Recall     F2   TP   FN
class_weight=balanced (current)       78283      11.3% 0.6612     0.1921  0.5289 0.3916 1236 1101
                          SMOTE      138826      50.0% 0.6413     0.1775  0.5379 0.3826 1257 1080
             RandomUnderSampler       17740      50.0% 0.6610     0.1931  0.5302 0.3929 1239 1098
             SMOTE + TomekLinks      138520      50.0% 0.6412     0.1776  0.5387 0.3830 1259 1078
  No imbalance handling (naive)       78283      11.3% 0.6613     0.4713  0.0175 0.0217   41 2296


# Class Imbalance Methods Comparison

## Context

The current model handles 11.5% positive class imbalance via 
`class_weight='balanced'`. Mentor feedback at the previous checkpoint 
specifically named SMOTE as worth testing. This notebook compares five 
imbalance-handling strategies on the final 30-feature model.

## Methods Compared

| Method | Family | Description |
|--------|--------|-------------|
| `class_weight='balanced'` | Reweighting | Current baseline |
| SMOTE | Oversampling | Synthetic minority oversampling |
| RandomUnderSampler | Undersampling | Random majority class removal |
| SMOTE + TomekLinks | Combined | Oversample + boundary cleaning |
| No handling (naive) | None | Reference baseline |

All methods evaluated on identical 30-feature L2 logistic regression, 
same `random_state`, default 0.5 threshold. Resampling applied to 
training data only; test set untouched to prevent leakage.

## Results

| Method | Train size | Train pos% | AUC | Recall | Precision | F2 |
|--------|-----------|-----------|-----|--------|-----------|-----|
| class_weight (current) | 78,283 | 11.3% | 0.6612 | 0.529 | 0.192 | 0.392 |
| SMOTE | 138,826 | 50.0% | 0.6413 | 0.538 | 0.178 | 0.383 |
| RandomUnderSampler | 17,740 | 50.0% | 0.6610 | 0.530 | 0.193 | 0.393 |
| SMOTE + TomekLinks | 138,520 | 50.0% | 0.6412 | 0.539 | 0.178 | 0.383 |
| No handling | 78,283 | 11.3% | 0.6613 | 0.018 | 0.471 | 0.022 |

## Key Findings

**1. SMOTE reduces AUC by ~0.02 compared to class_weight.**
SMOTE produces marginally higher recall at default threshold (0.539 vs 
0.529), but this is the result of a shifted decision boundary, not 
improved class discrimination. AUC, which is threshold-independent, 
drops measurably.

**2. RandomUnderSampler matches class_weight on AUC.**
Achieving equivalent discrimination by discarding ~60,000 training rows 
is generally a worse choice than keeping all data and reweighting.

**3. The "naive" baseline reveals why imbalance handling matters.**
Without any handling, the model defaults to predicting almost all 
patients as non-readmitted (recall 0.018, catching only 41 of 2,337 
actual readmissions). High AUC with near-zero recall is the signature 
of an unhandled imbalanced classifier.

## Why SMOTE Underperformed Here

Two likely reasons:

1. **Feature space is dominated by one-hot dummies.** SMOTE interpolates 
   between minority-class nearest neighbors. For binary features, this 
   produces non-binary values that don't correspond to real patients 
   (e.g., 0.5 for `discharge_disposition_id_22`). The synthetic samples 
   carry less clinical meaning than real ones.

2. **Imbalance is moderate, not severe.** SMOTE provides the largest 
   gains on highly imbalanced problems (1-5% positive class). At 11.5%, 
   the model already has sufficient minority-class signal to learn from 
   directly.

## Decision

**Retain `class_weight='balanced'` as the imbalance-handling method.** 
The current approach produces equivalent AUC to the best alternative 
(RandomUnderSampler) while using all available training data, and 
outperforms SMOTE by ~0.02 AUC.

The next improvement vector for this model is not imbalance handling — 
both alternatives match or underperform the current method. The 
remaining performance ceiling (AUC ~0.66 across all methods) reflects 
the linear model's discrimination limit, motivating exploration of 
non-linear models in the next iteration.

In [4]:
# Save the final-final state (post-imbalance comparison)
# The 30-feature model with class_weight='balanced' remains the chosen LR baseline
import joblib

final_lr_model = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
final_lr_model.fit(X_train_final, y_train)

joblib.dump(final_lr_model, '../data/final_lr_model_30features_classweight.pkl')

# Save the comparison results
comparison_df.to_csv('../data/imbalance_comparison.csv', index=False)
print("Saved final LR model and comparison results")

/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Saved final LR model and comparison results
